# Gaussian Splatting

## Overview COLMAP
Gaussian Splatting (GS) represents a scene as a set of 3D gaussians, which are
optimized by rendering them from the viewpoint of each input image and comparing
the result with that "ground truth" image. Once optimized, the Gaussians can be
rendered from any new viewpoint in real time, which is called novel view
synthesis. To optimize them (i.e. to move the gaussians to the right locations
with the right colors), we need the camera pose and camera parameters of every
input image. 

<img src="images/tomato_scene_to_gaussian_splats.png" style="width: 2000px;">

We estimate these with COLMAP, a Structure-from-Motion (SfM) and Multi-View
Stereo (MVS) pipeline that works on overlapping images, i.e. multiple images of
the same subject taken from different angles. GS only needs the SfM stage. The
COLMAP stages:

```text
Images
  ↓
Feature extraction
  ↓
Feature matching
  ↓
SfM (Structure-from-Motion)
  ├─ Camera intrinsics
  ├─ Camera poses
  ├─ Sparse 3D points
  └─ Bundle adjustment
  ↓
MVS (Multi-View Stereo)
  ├─ Depth maps
  ├─ Dense point cloud
  └─ Mesh
  ```

## Running COLMAP

Install COLMAP on Ubuntu with:

```bash
sudo apt update
sudo apt install colmap
```

Run the reconstruction with `colmap_commands.sh`. Set `IMAGES` in the script to
the directory containing the input images. During feature extraction, the script
uses:

```bash
--SiftExtraction.max_image_size 5000
```

This limits the largest image dimension used for SIFT feature extraction to 5000
pixels. Making it smaller speeds up extraction, but may reduce feature-matching
accuracy.

The mapper writes one folder per reconstruction:

```text
colmap_results/
├── 0/
├── 1/
└── ...
```

Normally, there is only `0/`. Additional folders mean that COLMAP split the
reconstruction into disconnected components. Each reconstruction contains the
same three files:

```text
cameras.bin
images.bin
points3D.bin
```

Only **registered** images appear in the reconstruction files. Images that
COLMAP could not localize are simply absent.

### Image undistortion

After reconstruction, the script runs `image_undistorter` on every
reconstruction. This removes lens distortion from the input images and converts
the camera model to a pinhole representation suitable for 3D Gaussian Splatting.

```text
colmap_results/
├── 0/
└── 0_undistorted/
    ├── images/
    ├── sparse/
    └── sparse_txt/
```

The `stereo/` directory and COLMAP helper scripts are removed because they are
only needed for dense MVS reconstruction, which is not used here. The
undistorted sparse model is also converted from `.bin` to `.txt` in
`sparse_txt/` for easier inspection; both formats contain the same
reconstruction data.

## Visualize COLMAP results

In [2]:
import numpy as np
import plotly.graph_objects as go

from utils.colmap import read_images_txt, read_points3D_txt

In [3]:
# Load colmap results.
MODEL = "colmap_results/0_undistorted/sparse_txt"

# 1. cameras.txt stores one camera's intrinsics per line: CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[].
# For the PINHOLE model, PARAMS[] = fx, fy, cx, cy: focal lengths and principal point in pixels.
### No visualization ###


# 2. images.txt stores each image over two lines:
#    IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME
#    POINTS2D[] as (X, Y, POINT3D_ID)
images = read_images_txt(f"{MODEL}/images.txt")
print(f"{len(images)} images")


# 3. points3D.txt stores one triangulated point per line:
#    POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[] as (IMAGE_ID, POINT2D_IDX)
points = read_points3D_txt(f"{MODEL}/points3D.txt")
print(f"{len(points.ids)} points")

166 images
25925 points


In [4]:
# Visualize colmap results. 

# --- Cameras ------------------------------------------------------------------
cams = [images[i] for i in sorted(images)]
centers = np.array([img.center for img in cams])  # Camera center in world coordinates: C = -R^T t.
view_dirs = np.array([img.R[2] for img in cams])  # Viewing direction: R^T (0, 0, 1).

# --- Points: drop outliers ----------------------------------------------------
mid = centers.mean(axis=0)
radius = np.linalg.norm(centers - mid, axis=1).max()
track_len = np.array([len(t) for t in points.tracks])
keep = (
    (points.errors < 1.0)                                  # low reprojection error
    & (track_len >= 3)                                     # seen in at least 3 images
    & (np.linalg.norm(points.xyz - mid, axis=1) < radius)  # inside the camera dome
)
xyz, rgb = points.xyz[keep], points.rgb[keep]

# --- Arrows: shaft from center to tip, cone head at the tip -------------------
length = 0.05 * np.ptp(centers, axis=0).max()
tips = centers + length * view_dirs
shafts = np.full((3 * len(cams), 3), None)  # center, tip, None break per camera
shafts[0::3], shafts[1::3] = centers, tips

# --- Plot ---------------------------------------------------------------------
fig = go.Figure([
    go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode="markers",
        marker=dict(size=1, color=[f"rgb({r},{g},{b})" for r, g, b in rgb]),
        name="points",
        hoverinfo="skip",
    ),
    go.Scatter3d(
        x=centers[:, 0], y=centers[:, 1], z=centers[:, 2],
        mode="markers",
        marker=dict(size=3, color="red"),
        text=[img.name for img in cams],
        hoverinfo="text",
        name="cameras",
    ),
    go.Scatter3d(
        x=shafts[:, 0], y=shafts[:, 1], z=shafts[:, 2],
        mode="lines",
        line=dict(color="red", width=3),
        hoverinfo="skip",
        showlegend=False,
    ),
    go.Cone(
        x=tips[:, 0], y=tips[:, 1], z=tips[:, 2],
        u=view_dirs[:, 0], v=view_dirs[:, 1], w=view_dirs[:, 2],
        anchor="tip",
        sizemode="absolute",
        sizeref=0.3 * length,
        colorscale=[[0, "red"], [1, "red"]],
        showscale=False,
        hoverinfo="skip",
    ),
])
fig.update_layout(
    scene={
        "aspectmode": "data",
        "dragmode": "orbit",
        "xaxis": {"showspikes": False},
        "yaxis": {"showspikes": False},
        "zaxis": {"showspikes": False},
    },
    scene_camera={"eye": {"x": 0.55, "y": 0.55, "z": 0.55}},
    height=800,
)
fig.show()